In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

rfm_base = pd.read_csv('/Users/Sankalp/Downloads/rfm_base.csv')

print(f"Loaded {len(rfm_base):,} customers")
# RFM Scoring:

rfm = rfm_base.copy()

rfm['R_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1,2,3,4,5])

rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print(rfm[['customer_unique_id','recency_days','frequency','monetary',
           'R_score','F_score','M_score','RFM_score']].head())

# Segmentation:

def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Potential Loyalist'

rfm['segment'] = rfm.apply(assign_segment, axis=1)
rfm['churned'] = rfm['recency_days'] > 90

print(rfm['segment'].value_counts())
print(f"\nChurned: {rfm['churned'].sum():,} ({rfm['churned'].mean()*100:.1f}%)")


# Export :

rfm.to_csv('rfm_segmented.csv', index=False)
print("Done — rfm_segmented.csv ready for Power BI")


Loaded 1,000 customers
                 customer_unique_id  recency_days  frequency  monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2           111          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           114          1     27.19   
2  0000f46a3911fa3c0805444483337064           537          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078           321          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255           288          1    196.89   

   R_score  F_score  M_score  RFM_score  
0        4        1        4          9  
1        4        1        1          6  
2        1        1        3          5  
3        2        1        1          4  
4        2        1        4          7  
segment
Loyal                 289
At Risk               246
Potential Loyalist    177
New Customer          168
Champion               65
Lost                   55
Name: count, dtype: int64

Churned: 800 (80.0%)
Done — rfm_segmented.csv ready for Power BI
